In [1]:
!pip install -q transformers torch datasets accelerate tqdm bitsandbytes huggingface-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.2 MB/s eta 0:00:00


In [3]:
from huggingface_hub import login
login()

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct" 
SYSTEM_PROMPT = "You are a helpful AI assistant. Be concise and friendly."

USE_HISTORY = True                 
CONTEXT_METHOD = "hybrid"          

MAX_NEW_TOKENS = 256
MAX_CONTEXT_TOKENS = 2048          
TOKEN_THRESHOLD_RATIO = 0.8        
SLIDING_WINDOW_TURNS = 8           
FIXED_WINDOW_MESSAGES = 12         
SUMMARY_MAX_NEW_TOKENS = 128       
TEMPERATURE = 0.7
TOP_P = 0.9

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Device:", device)
print("Loading tokenizer & model (may take a minute)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, device_map="auto", low_cpu_mem_usage=True)
model.eval()
print("Model loaded on", model.device)
print()

def format_messages(messages):
    """Simple [ROLE]: content formatting used in prompts."""
    return "\n".join(f"[{m['role'].upper()}]: {m['content']}" for m in messages) + "\n[ASSISTANT]:"

def count_tokens_text(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

def count_tokens_messages(messages):
    return count_tokens_text(format_messages(messages))

def show_context_stats(messages):
    total = count_tokens_messages(messages)
    print(f"[Context tokens] {total}/{MAX_CONTEXT_TOKENS} ({total/MAX_CONTEXT_TOKENS:.2%})")
    return total

def fixed_window(history, max_messages=FIXED_WINDOW_MESSAGES):
    """Keep system + last (max_messages-1) items."""
    system = [m for m in history if m["role"] == "system"]
    non_sys = [m for m in history if m["role"] != "system"]
    if len(non_sys) <= max_messages - 1:
        return history
    kept = non_sys[-(max_messages - 1):]
    return system + kept

def sliding_window(history, keep_turns=SLIDING_WINDOW_TURNS):
    """Keep system + last keep_turns messages (counting both user and assistant messages)."""
    system = [m for m in history if m["role"] == "system"]
    non_sys = [m for m in history if m["role"] != "system"]
    kept = non_sys[-keep_turns:]
    return system + kept

def token_truncate(history, max_tokens=MAX_CONTEXT_TOKENS):
    """Token-aware truncation: remove oldest non-system messages until under budget."""
    system = [m for m in history if m["role"] == "system"]
    non_sys = [m for m in history if m["role"] != "system"]
    kept = []
    total = count_tokens_text(format_messages(system)) if system else 0
    for m in reversed(non_sys):
        t = count_tokens_text(f"[{m['role'].upper()}]: {m['content']}")
        if total + t + 16 > max_tokens:
            break
        kept.insert(0, m)
        total += t
    return system + kept

def summarize_older(history, keep_recent=SLIDING_WINDOW_TURNS, max_summary_tokens=SUMMARY_MAX_NEW_TOKENS):
    """
    Summarize older messages (everything except last keep_recent messages)
    Returns: system + [summary_msg] + recent
    """
    system = [m for m in history if m["role"] == "system"]
    non_sys = [m for m in history if m["role"] != "system"]
    if len(non_sys) <= keep_recent:
        return history  

    older = non_sys[:-keep_recent]
    recent = non_sys[-keep_recent:]

    older_text = "\n".join(f"[{m['role'].upper()}]: {m['content']}" for m in older)
    summary_prompt = (
        "You are a concise summarizer. Produce a short factual summary (bullet points or a paragraph) "
        "that captures the important facts, preferences, and decisions from the conversation below. "
        "Do not invent details.\n\n"
        "Conversation to summarize:\n" + older_text + "\n\nSummary:"
    )

    enc = tokenizer(summary_prompt, return_tensors="pt", truncation=True).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_summary_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    summary = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if not summary:
        summary = " ".join(m["content"] for m in older)[:800]

    summary_msg = {"role": "assistant", "content": "[SUMMARY OF EARLIER CONVERSATION]: " + summary}
    new_history = system + [summary_msg] + recent

    if count_tokens_messages(new_history) > MAX_CONTEXT_TOKENS:
        return token_truncate(new_history, max_tokens=MAX_CONTEXT_TOKENS)
    return new_history

def apply_hybrid_management(history):
    """
    Hybrid strategy:
      - Always keep system prompt.
      - Keep sliding window of recent turns.
      - If tokens > threshold, summarize older messages into one summary.
    """
    h = sliding_window(history, keep_turns=SLIDING_WINDOW_TURNS)
    tokens = count_tokens_messages(h)
    if tokens < TOKEN_THRESHOLD_RATIO * MAX_CONTEXT_TOKENS:
        return h  
    return summarize_older(history, keep_recent=SLIDING_WINDOW_TURNS)

def prepare_inputs_for_generation(history):
    """
    Prefer tokenizer.apply_chat_template when available.
    Return dict with input_ids and attention_mask on model.device.
    """
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            enc = tokenizer.apply_chat_template(history, add_generation_prompt=True, return_tensors="pt")
            input_ids = enc["input_ids"].to(model.device)
            attention_mask = enc["attention_mask"].to(model.device) if "attention_mask" in enc else torch.ones_like(input_ids).to(model.device)
            return {"input_ids": input_ids, "attention_mask": attention_mask}
        except Exception:
            pass

    prompt = format_messages(history)
    enc = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
    input_ids = enc["input_ids"].to(model.device)
    attention_mask = enc["attention_mask"].to(model.device) if "attention_mask" in enc else torch.ones_like(input_ids).to(model.device)
    return {"input_ids": input_ids, "attention_mask": attention_mask}

def truncate_to_first_assistant_reply(text):
    markers = ["\n[USER]:", "\n[ASSISTANT]:", "[USER]:", "[ASSISTANT]:"]
    first_pos = None
    for m in markers:
        p = text.find(m)
        if p != -1 and (first_pos is None or p < first_pos):
            first_pos = p
    if first_pos is not None:
        return text[:first_pos].strip()
    return text.strip()

def generate_response_with_history(history, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    tensors = prepare_inputs_for_generation(history)
    input_ids = tensors["input_ids"]
    attention_mask = tensors["attention_mask"]
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][input_ids.shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    reply = truncate_to_first_assistant_reply(raw)
    reply = reply.replace("[ASSISTANT]:", "").replace("[USER]:", "").strip()
    return reply

print("Chat ready — type 'exit' to stop.\n")
chat_history = [{"role": "system", "content": SYSTEM_PROMPT}]

while True:
    try:
        user_text = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nExiting.")
        break

    if not user_text:
        continue
    if user_text.lower() in ("exit", "quit", "q"):
        print("Goodbye!")
        break

    full_history = chat_history + [{"role": "user", "content": user_text}]

    if not USE_HISTORY or CONTEXT_METHOD == "none":
        active_history = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_text}]
    elif CONTEXT_METHOD == "fixed_window":
        full_history = chat_history + [{"role": "user", "content": user_text}]
        active_history = fixed_window(full_history, max_messages=FIXED_WINDOW_MESSAGES)
    elif CONTEXT_METHOD == "sliding_window":
        full_history = chat_history + [{"role": "user", "content": user_text}]
        active_history = sliding_window(full_history, keep_turns=SLIDING_WINDOW_TURNS)
    elif CONTEXT_METHOD == "hybrid":
        full_history = chat_history + [{"role": "user", "content": user_text}]
        active_history = apply_hybrid_management(full_history)
    else:
        full_history = chat_history + [{"role": "user", "content": user_text}]
        active_history = token_truncate(full_history, max_tokens=MAX_CONTEXT_TOKENS)

    total_tokens = show_context_stats(active_history)
    if total_tokens > MAX_CONTEXT_TOKENS:
        print("[Warning] Active history exceeded MAX_CONTEXT_TOKENS — applying token truncation fallback.")
        active_history = token_truncate(active_history, max_tokens=MAX_CONTEXT_TOKENS)

    start = time.time()
    reply = generate_response_with_history(active_history)
    latency = time.time() - start

    print("Assistant:", reply)
    print(f"(latency: {latency:.2f}s)\n")
    if USE_HISTORY:
        chat_history.append({"role": "user", "content": user_text})
        chat_history.append({"role": "assistant", "content": reply})
        if len([m for m in chat_history if m["role"] != "system"]) > 200:
            print("[Note] Compressing long-lived conversation to save memory.")
            chat_history = summarize_older(chat_history, keep_recent=SLIDING_WINDOW_TURNS)

Device: cuda
Loading tokenizer & model (may take a minute)...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Model loaded on cuda:0

Chat ready — type 'exit' to stop.

You: Hey, how are you?
[Context tokens] 29/2048 (1.42%)
Assistant: I'm doing well, thanks for asking. How can I assist you today?
(latency: 0.09s)

You: What is cricket?
[Context tokens] 57/2048 (2.78%)
Assistant: Cricket is a popular team sport played with a bat and a ball. It's a bit like a combination of baseball and association football (soccer). The objective is to score runs by hitting the ball with a bat and running between two sets of three stumps (called wickets) while the opposing team tries to stop them by getting the batsmen out.

The game is played with 11 players on each team, and it's usually played on a rectangular field with a flat surface, such as grass or turf. The team with the most runs at the end of the game wins.

Cricket is played all over the world, with different versions and rules, but the basic concept remains the same. It's a fun and exciting game that requires skill, strategy, and teamwork.

Are yo